# Simulation der Lückenlängen

Eine Urne enthält $w$ weiße und $s$ schwarze Kugeln. Alle
$N=w+s$ Kugeln werden zufällig angeordnet. Die $w$ weißen Kugeln legen
$w+1$ Lücken fest:

- vor der ersten weißen Kugel,
- zwischen je zwei weißen Kugeln,
- nach der letzten weißen Kugel.

Auch leere Lücken sind zugelassen. Für ihre Längen
$G_0,\ldots,G_w$ gilt in jeder Anordnung

$$G_0+G_1+\dots+G_w=s.$$

Das Notebook simuliert die Verteilung einer ausgewählten Lücke und
bestimmt für alle Lücken den empirischen Mittelwert, die empirische
Varianz und die empirische Standardabweichung. Für $w=2$ und $s=3$
können die exakten Werte unmittelbar mit der Simulation verglichen
werden.

## 1. Eingaben

Nur die Werte in der folgenden Zelle müssen geändert werden. Mit `seed = 42` ist die Simulation reproduzierbar.

In [ ]:
# Anzahl der Kugeln
w = 2                    # weiße Kugeln
s = 3                    # schwarze Kugeln

# Einstellungen der Simulation
anzahl_simulationen = 10_000
luecke_anzeigen = 0      # 0, 1, ..., w
seed = 42

# Grafik bei Bedarf zusätzlich speichern
grafik_speichern = False
dateiname = "Simulation_Luecken.pdf" 

## 2. Lücken in einer einzelnen Anordnung bestimmen

Die Funktion zählt zunächst die schwarzen Kugeln vor der ersten weißen Kugel. Nach jeder weißen Kugel beginnt die nächste Lücke. Aufeinanderfolgende weiße Kugeln erzeugen daher eine leere Lücke.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def bestimme_luecken(anordnung):
    """Bestimmt die Lückenlängen einer Anordnung aus 'W' und 'S'."""
    anzahl_weiss = sum(kugel == "W" for kugel in anordnung)
    luecken = np.zeros(anzahl_weiss + 1, dtype=int)
    aktuelle_luecke = 0

    for kugel in anordnung:
        if kugel == "W":
            aktuelle_luecke += 1
        else:
            luecken[aktuelle_luecke] += 1

    return luecken


# Eine einzelne, reproduzierbare Anordnung als Beispiel
rng_beispiel = np.random.default_rng(seed)
grundanordnung = np.array(["W"] * w + ["S"] * s)
beispiel = rng_beispiel.permutation(grundanordnung)
luecken_beispiel = bestimme_luecken(beispiel)

print("Anordnung: ", " ".join(beispiel))
print("Lücken:    ", luecken_beispiel.tolist())
print("Kontrolle: ", f"{luecken_beispiel.sum()} schwarze Kugeln")

## 3. Viele zufällige Anordnungen erzeugen

Bei jeder Wiederholung werden alle Kugeln neu gemischt und anschließend sämtliche Lückenlängen bestimmt.

In [ ]:
def simuliere_luecken(w, s, anzahl_simulationen, seed=42):
    """Simuliert Lückenlängen für zufällige Farbanordnungen."""
    if not isinstance(w, (int, np.integer)) or w < 1:
        raise ValueError("w muss eine positive ganze Zahl sein.")
    if not isinstance(s, (int, np.integer)) or s < 0:
        raise ValueError("s muss eine nichtnegative ganze Zahl sein.")
    if not isinstance(anzahl_simulationen, (int, np.integer)) or anzahl_simulationen < 1:
        raise ValueError("anzahl_simulationen muss eine positive ganze Zahl sein.")

    rng = np.random.default_rng(seed)
    grundanordnung = np.array(["W"] * w + ["S"] * s)
    ergebnisse = np.empty((anzahl_simulationen, w + 1), dtype=int)

    for nr in range(anzahl_simulationen):
        anordnung = rng.permutation(grundanordnung)
        ergebnisse[nr] = bestimme_luecken(anordnung)

    return ergebnisse


ergebnisse = simuliere_luecken(
    w=w,
    s=s,
    anzahl_simulationen=anzahl_simulationen,
    seed=seed,
)

if not 0 <= luecke_anzeigen <= w:
    raise ValueError("luecke_anzeigen muss zwischen 0 und w liegen.")

theoretischer_mittelwert = s / (w + 1)
theoretische_varianz = (
    w * s * (w + s + 1)
    / ((w + 1) ** 2 * (w + 2))
)
theoretische_standardabweichung = np.sqrt(theoretische_varianz)

empirische_mittelwerte = ergebnisse.mean(axis=0)
empirische_varianzen = ergebnisse.var(axis=0)
empirische_standardabweichungen = ergebnisse.std(axis=0)

tabelle = pd.DataFrame({
    "Lücke": [f"G_{i}" for i in range(w + 1)],
    "E empirisch": empirische_mittelwerte,
    "V empirisch": empirische_varianzen,
    "σ empirisch": empirische_standardabweichungen,
    "E theoretisch": theoretischer_mittelwert,
    "V theoretisch": theoretische_varianz,
    "σ theoretisch": theoretische_standardabweichung,
})

tabelle.round(4)

## 4. Empirische Verteilung und Kenngrößen

Das Stabdiagramm zeigt die empirische Verteilung der ausgewählten
Lücke. Die senkrechten Strecken um die blauen Punkte sind punktweise
$95\,\%$-Wilson-Konfidenzintervalle für die einzelnen
Wahrscheinlichkeiten $p_k=P(G_i=k)$. Die roten Kreuze markieren zum
Vergleich die exakten Wahrscheinlichkeiten. Im Diagramm werden stets
auch Erwartungswert, Varianz und Standardabweichung angegeben.

In [ ]:
plt.rcParams.update({
    "font.size": 11,
    "axes.spines.top": True,
    "axes.spines.right": True,
})

from math import comb
from statistics import NormalDist

werte = np.arange(s + 1)
haeufigkeiten = np.bincount(
    ergebnisse[:, luecke_anzeigen],
    minlength=s + 1,
)
empirische_wahrscheinlichkeiten = haeufigkeiten / anzahl_simulationen

# Exakte Randverteilung einer beliebigen Lücke
anzahl_aller_anordnungen = comb(w + s, w)
exakte_wahrscheinlichkeiten = np.array([
    comb(s - k + w - 1, w - 1) / anzahl_aller_anordnungen
    for k in werte
])

# Punktweise 95-%-Wilson-Konfidenzintervalle für p_k = P(G_i = k)
konfidenzniveau = 0.95
z = NormalDist().inv_cdf(1 - (1 - konfidenzniveau) / 2)
nenner = 1 + z**2 / anzahl_simulationen
wilson_mitten = (
    empirische_wahrscheinlichkeiten
    + z**2 / (2 * anzahl_simulationen)
) / nenner
wilson_halbbreiten = z / nenner * np.sqrt(
    empirische_wahrscheinlichkeiten
    * (1 - empirische_wahrscheinlichkeiten)
    / anzahl_simulationen
    + z**2 / (4 * anzahl_simulationen**2)
)
wilson_unten = wilson_mitten - wilson_halbbreiten
wilson_oben = wilson_mitten + wilson_halbbreiten

fig, ax = plt.subplots(figsize=(8.2, 5.2))

# Empirische Verteilung: Stäbe mit Punkten
ax.vlines(
    werte,
    0,
    empirische_wahrscheinlichkeiten,
    color="tab:blue",
    linewidth=2.2,
)
ax.plot(
    werte,
    empirische_wahrscheinlichkeiten,
    "o",
    color="tab:blue",
    markersize=6,
)

# Punktweise Wilson-Konfidenzintervalle als senkrechte Strecken
ax.errorbar(
    werte,
    empirische_wahrscheinlichkeiten,
    yerr=np.vstack((
        empirische_wahrscheinlichkeiten - wilson_unten,
        wilson_oben - empirische_wahrscheinlichkeiten,
    )),
    fmt="none",
    ecolor="0.25",
    elinewidth=1.2,
    capsize=4,
    capthick=1.2,
)

# Exakte Wahrscheinlichkeiten zum Vergleich
ax.plot(
    werte,
    exakte_wahrscheinlichkeiten,
    "x",
    color="tab:red",
    markersize=8,
    markeredgewidth=2,
)

emp_E = empirische_mittelwerte[luecke_anzeigen]
emp_V = empirische_varianzen[luecke_anzeigen]
emp_sigma = empirische_standardabweichungen[luecke_anzeigen]

ax.text(
    0.97,
    0.95,
    rf"empirisch: $E={emp_E:.4f}$" + "\n"
    + rf"$V={emp_V:.4f}$, $\sigma={emp_sigma:.4f}$" + "\n\n"
    + rf"theoretisch: $E={theoretischer_mittelwert:.4f}$" + "\n"
    + rf"$V={theoretische_varianz:.4f}$, $\sigma={theoretische_standardabweichung:.4f}$",
    ha="right",
    va="top",
    transform=ax.transAxes,
    bbox={"facecolor": "white", "edgecolor": "0.75", "alpha": 0.9, "pad": 5},
)

ax.set_title(rf"Verteilung der Lückenlänge $G_{{{luecke_anzeigen}}}$")
ax.set_xlabel("Anzahl schwarzer Kugeln in der Lücke")
ax.set_ylabel("Wahrscheinlichkeit")
ax.set_xticks(werte)
ax.set_ylim(bottom=0)
fig.text(
    0.5,
    0.02,
    "Blau: Simulation; Strecken: punktweise 95-%-Wilson-Intervalle; "
    "rote Kreuze: exakte Wahrscheinlichkeiten.",
    ha="center",
    va="bottom",
    fontsize=9,
)

fig.suptitle(
    rf"$w={w}$ weiße und $s={s}$ schwarze Kugeln",
    fontsize=13,
)
fig.tight_layout(rect=(0, 0.08, 1, 1))

if grafik_speichern:
    fig.savefig(dateiname, bbox_inches="tight")

plt.show()

## Beobachtung und Einordnung

Für $w=2$ und $s=3$ ist die exakte Verteilung einer beliebigen Lücke besonders übersichtlich:

$$P(G_i=0)=0{,}4,\quad P(G_i=1)=0{,}3,\quad P(G_i=2)=0{,}2,\quad P(G_i=3)=0{,}1.$$

Daraus folgen

$$\operatorname{E}(G_i)=1,\qquad \operatorname{V}(G_i)=1,\qquad \sigma(G_i)=1.$$

Die fallenden Wahrscheinlichkeiten machen die Rechtsschiefe sichtbar: Leere und kurze Lücken sind häufig; längere Lücken treten seltener auf, liegen dann aber deutlich oberhalb des Erwartungswertes. Diese größeren Abweichungen gehen quadratisch in die Varianz ein.

Für größere Werte von $w$ und $s$ wird das vollständige Auflisten der Anordnungen und die exakte Berechnung von Hand schnell sperrig. Die Simulation ist deshalb keine bloße Notlösung: Sie macht die Verteilung sichtbar und liefert Näherungen für Erwartungswert, Varianz und Standardabweichung. Die Symmetrie erklärt unabhängig davon, warum alle Lücken dieselbe Verteilung besitzen.

Die punktweisen Wilson-Konfidenzintervalle quantifizieren zusätzlich die Genauigkeit, mit der die einzelnen Wahrscheinlichkeiten $p_k=P(G_i=k)$ durch die Simulation geschätzt werden. Die hier bekannten exakten Werte dienen mit den roten Kreuzen zur Kontrolle; in wissenschaftlichen Anwendungen sind gerade diese Zielwerte häufig unbekannt.

Allgemein gilt

$$\operatorname{E}(G_i)=\frac{s}{w+1},\qquad
\operatorname{V}(G_i)=
\frac{ws(w+s+1)}{(w+1)^2(w+2)}.$$

Die Herleitung der Varianzformel ist wesentlich aufwendiger als das Symmetrieargument für den Erwartungswert. Für den schulischen Erkenntnisweg ist es daher oft ergiebiger, die Streuung zunächst zu simulieren und aus der Gestalt der Verteilung zu deuten.